# 2. Naive next-day prediction

Predict **entire next day** using the **naive method**: for each (hour, frequency, threshold, class), prediction = **same value at that hour the previous day**.

- **Error:** computed over the entire day (all hours × freqs × thresholds) → **one MAE and one RMSE per day**.
- **Testing only:** previous day is loaded from either training or testing dir. No training metrics shown.
- **Final visualization:** dropdown for **class**; show testing MAE/RMSE per day. **Final results table:** MAE (Naive) per class.

In [31]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Optional
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

In [32]:
# Path to final data. Look in organized/ or work_dir/.
work_dir = None
for candidate in [Path("organized"), Path("../organized"), Path("work_dir"), Path("../work_dir")]:
    fd = candidate / "final"
    if fd.exists():
        work_dir = candidate
        break
if work_dir is None:
    raise FileNotFoundError('Final dir not found. Tried: organized/final, ../organized/final, work_dir/final, ../work_dir/final')
final_dir = work_dir / "final"
training_dir = final_dir / "training"
testing_dir = final_dir / "testing"

In [33]:
def load_final_parquet(split_dir: Path, band: str, date_yyyymmdd: str) -> Optional[pd.DataFrame]:
    """Load final_<date>.parquet for a given split and band. Returns None if missing."""
    path = split_dir / band / f"final_{date_yyyymmdd}.parquet"
    if not path.exists():
        return None
    return pd.read_parquet(path)


def get_dates_and_bands(split_dir: Path):
    """Return sorted list of (date_yyyymmdd, band) and list of unique bands."""
    dates = set()
    bands = []
    for band_dir in sorted(split_dir.iterdir()):
        if not band_dir.is_dir():
            continue
        if band_dir.name not in bands:
            bands.append(band_dir.name)
        for p in band_dir.glob("final_*.parquet"):
            d = p.stem.replace("final_", "")
            dates.add(d)
    return sorted(dates), sorted(set(bands))

In [34]:
# Discover bands (classes) from testing (and training for resolving previous-day)
_, bands_training = get_dates_and_bands(training_dir)
_, bands_testing = get_dates_and_bands(testing_dir)
class_options = sorted(set(bands_training) | set(bands_testing))
testing_dates = get_dates_and_bands(testing_dir)[0]

### Naive prediction (testing only)

For each **testing** date: prediction = previous day's value at same (hour, freq, threshold). Previous day is loaded from **training or testing** dir. MAE and RMSE per day over the entire day.

In [35]:
def date_to_prev_yyyymmdd(date_yyyymmdd: str) -> str | None:
    """Return previous calendar day in YYYYMMDD. Simple string logic for contiguous dates."""
    from datetime import datetime, timedelta
    try:
        dt = datetime.strptime(date_yyyymmdd, "%Y%m%d")
        prev = dt - timedelta(days=1)
        return prev.strftime("%Y%m%d")
    except Exception:
        return None


def compute_naive_errors_one_band(split_dir: Path, band: str, dates: list[str], prev_day_dirs: Optional[list] = None) -> pd.DataFrame:
    """
    For each date in dates that has a previous day, load current and previous,
    merge on (hour, freq_center_ghz, threshold_dbm), compute MAE and RMSE over the full day.
    prev_day_dirs: where to look for previous-day data (default [split_dir]). Use [training_dir, testing_dir]
    for testing so 02/06 can use 02/05 from training.
    Returns a DataFrame with columns: date_yyyymmdd, MAE, RMSE.
    """
    if prev_day_dirs is None:
        prev_day_dirs = [split_dir]
    key_cols = ["hour", "freq_center_ghz", "threshold_dbm"]
    rows = []
    for date_curr in dates:
        date_prev = date_to_prev_yyyymmdd(date_curr)
        if date_prev is None:
            continue
        df_curr = load_final_parquet(split_dir, band, date_curr)
        df_prev = None
        for d in prev_day_dirs:
            df_prev = load_final_parquet(d, band, date_prev)
            if df_prev is not None:
                break
        if df_curr is None or df_prev is None:
            continue
        df_prev = df_prev.rename(columns={"au_pct": "au_pct_prev"})
        merge = df_curr.merge(
            df_prev[key_cols + ["au_pct_prev"]],
            on=key_cols,
            how="inner",
        )
        if merge.empty:
            continue
        err = merge["au_pct"] - merge["au_pct_prev"]
        mae = np.abs(err).mean()
        rmse = np.sqrt((err ** 2).mean())
        rows.append({"date_yyyymmdd": date_curr, "MAE": mae, "RMSE": rmse})
    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=["date_yyyymmdd", "MAE", "RMSE"])


def compute_testing_mae_rmse_one_band(band: str) -> tuple:
    """
    Testing: for each testing date we need previous day. Previous day may be in training (last day)
    or in testing. Returns (MAE, RMSE) over all testing data.
    """
    key_cols = ["hour", "freq_center_ghz", "threshold_dbm"]
    all_errs = []
    for date_curr in testing_dates:
        date_prev = date_to_prev_yyyymmdd(date_curr)
        if date_prev is None:
            continue
        df_curr = load_final_parquet(testing_dir, band, date_curr)
        if df_curr is None:
            continue
        df_prev = load_final_parquet(training_dir, band, date_prev)
        if df_prev is None:
            df_prev = load_final_parquet(testing_dir, band, date_prev)
        if df_prev is None:
            continue
        df_prev = df_prev.rename(columns={"au_pct": "au_pct_prev"})
        merge = df_curr.merge(
            df_prev[key_cols + ["au_pct_prev"]],
            on=key_cols,
            how="inner",
        )
        if merge.empty:
            continue
        err = merge["au_pct"] - merge["au_pct_prev"]
        all_errs.extend(err.tolist())
    if not all_errs:
        return np.nan, np.nan
    arr = np.array(all_errs)
    return float(np.abs(arr).mean()), float(np.sqrt((arr ** 2).mean()))

In [36]:
# Compute testing per-day MAE/RMSE and overall testing MAE/RMSE per band (previous day from train or test)
results = {}
for band in class_options:
    test_errors = compute_naive_errors_one_band(testing_dir, band, testing_dates, prev_day_dirs=[training_dir, testing_dir])
    test_mae, test_rmse = compute_testing_mae_rmse_one_band(band)
    results[band] = {
        "testing_per_day": test_errors,
        "testing_MAE": test_mae,
        "testing_RMSE": test_rmse,
    }

# Cumulative table: Class, MAE, RMSE
final_results = pd.DataFrame([
    {"Class": band, "MAE": results[band]["testing_MAE"], "RMSE": results[band]["testing_RMSE"]}
    for band in class_options
])
print("Naive next-day prediction — Final results (testing data)")
display(final_results.round(4))

Naive next-day prediction — Final results (testing data)


,Class,MAE,RMSE
0,195MHz,2.4381,9.0981
1,2441MHz,5.9981,8.6974
2,3765MHz,9.1502,15.6977
3,539MHz,3.0888,10.9003
4,5500MHz,3.6693,7.2692
5,915MHz,4.9137,7.3706


### Per-class view: dropdown for class — testing MAE/RMSE per day

In [37]:
class_dropdown = widgets.Dropdown(
    options=class_options,
    value=class_options[0] if class_options else None,
    description="Class:",
    style={"description_width": "50px"},
)
out = widgets.Output()


def update_naive_viz(class_band):
    with out:
        clear_output(wait=True)
        if class_band not in results:
            print(f"No results for class {class_band}")
            return
        r = results[class_band]
        test_df = r["testing_per_day"]
        test_mae = r["testing_MAE"]

        # Table: testing per-day MAE, RMSE only
        if not test_df.empty:
            test_display = test_df.copy()
            test_display["date"] = test_display["date_yyyymmdd"].str[:4] + "-" + test_display["date_yyyymmdd"].str[4:6] + "-" + test_display["date_yyyymmdd"].str[6:8]
            print(f"Naive next-day prediction — Class: {class_band} (testing only)")
            display(test_display[["date", "MAE", "RMSE"]].round(4))
        print(f"Testing MAE (all testing days): {test_mae:.4f}")

        # Bar chart: MAE per testing date
        fig = go.Figure()
        if not test_df.empty:
            test_dates_dash = [f"{d[:4]}-{d[4:6]}-{d[6:8]}" for d in test_df["date_yyyymmdd"]]
            fig.add_trace(go.Bar(x=test_dates_dash, y=test_df["MAE"], name="MAE (per day)", marker_color="coral"))
        fig.update_layout(
            title=f"Naive next-day MAE by date — {class_band}",
            xaxis_title="Date",
            yaxis_title="MAE",
            height=400,
        )
        fig.show()


widgets.interactive_output(update_naive_viz, {"class_band": class_dropdown})
display(widgets.HBox([class_dropdown]), out)
update_naive_viz(class_dropdown.value)

Output()

### Predicted vs Actual heatmaps

Pick a **class** and **testing date**. Left: actual AU (%) for that day. Right: naive prediction (previous day same hour/freq). Same colorscale for comparison.